# Paper figures

Builds every figure and table used in the paper from **saved** experiment outputs. No simulation runs here: rerun `run_mpc.ipynb` / `run_irf.ipynb` first, then rerun this notebook.

All files are written to `data/output_data/paper-figures/`.

| Figure | Source run | Status |
|---|---|---|
| Impact cash MPC distributions (12 panels) | `run_mpc.ipynb`, `write_paper_figures` | in paper |
| MPC by household group (3 groups figure, 8 groups table, KVW robustness) | `run_mpc.ipynb` panel + `src/household_classes.py` | new |
| Aggregate MPC table | `run_mpc.ipynb`, `paper_mpc_summary_filtered.csv` | in paper |
| IRF: monetary policy +100bp | `run_irf.ipynb` | in paper |
| IRF: unemployment +2pp | `run_irf.ipynb` | candidate (C response under review) |
| IRF: one-off transfer, total uncertainty | `run_irf.ipynb` | add to `PAPER_IRF_SHOCKS` once checked |

Excluded until diagnosed: income-tax IRF (wrong sign), individual/common uncertainty IRFs.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from src.household_classes import (
    GROUP_LABELS,
    HTM_CLASSES,
    THREE_GROUPS,
    classify_balance_sheet_groups,
    classify_households,
)
from src.mpc_paper_figures import write_paper_figures
from src.visual_helpers import _prepare_irf_plot_panel

MPC_DIR = Path("data/output_data/mpc-experiment/analysis")
IRF_DIR = Path("data/output_data/irf-experiment/analysis")
OUT = Path("data/output_data/paper-figures")
OUT.mkdir(parents=True, exist_ok=True)
SHOW_TRANSFER_WEIGHTED_MEAN = False  # equals the household mean: transfers are equal within a seed

# MPC measure used in every paper figure and table.
# Cash MPC = change in household cash consumption spending, i.e. goods INCLUDING VAT (+ cash rent,
# unchanged on impact), deflated by CPI, per unit of real transfer. The transfer is cash, so the
# numerator must be cash too. The VAT-exclusive goods measure (real_mpc_impact) is lower by the
# factor 1/(1+VAT) = 1/1.13: a household spending the whole transfer on goods would show 0.885.
MPC_IMPACT = "cash_real_mpc_impact"
MPC_CUMULATIVE = {
    "2 quarters": "cash_real_cmpc_2p",
    "3 quarters": "cash_real_cmpc_3p",
    "4 quarters": "cash_real_cmpc_4q",
}

## 1. Impact MPC distributions
**Measure: cash MPC** (`cash_real_mpc_impact`): real change in cash consumption spending (goods including VAT) in the transfer quarter, per unit of real transfer. Filtered household-seed panel; boxes = distribution, dots = household means.

In [ ]:
# Same panel run_mpc.ipynb passes to write_paper_figures (filtered when apply_mpc_filters=True).
panel_path = MPC_DIR / "household_mpc_panel_filtered.csv"
if not panel_path.exists():
    panel_path = MPC_DIR / "household_mpc_panel.csv"
mpc_panel = pd.read_csv(panel_path)
# write_paper_figures plots the column real_mpc_impact: hand it the cash MPC under that name.
figure_panel = mpc_panel.assign(real_mpc_impact=mpc_panel[MPC_IMPACT])
try:
    write_paper_figures(figure_panel, OUT, show_transfer_weighted_mean=SHOW_TRANSFER_WEIGHTED_MEAN)
except TypeError:  # older signature without the marker switch
    write_paper_figures(figure_panel, OUT)
display(Image(filename=str(OUT / "paper_impact_mpc.png")))
pd.read_csv(OUT / "paper_figure_group_means.csv").round(3)

## 2. Aggregate MPC table
Cash MPC, transfer-weighted (ΣΔC / Σtransfer) over **all** households in the filtered panel, for impact and cumulative horizons. No labour-income identification filter: cumulative MPCs include the general-equilibrium feedback of the transfer on labour income (net about +1% of the transfer). SE: standard deviation of per-seed MPCs / √seeds.

In [ ]:
def weighted_mpc(panel, column):
    d = panel[np.isfinite(panel[column]) & np.isfinite(panel.real_shock_amount)]
    w = d.real_shock_amount
    per_seed = (d[column] * w).groupby(d.seed).sum() / w.groupby(d.seed).sum()
    return pd.Series(
        {
            "MPC": float((d[column] * w).sum() / w.sum()),
            "SE (seed)": float(per_seed.std(ddof=1) / np.sqrt(len(per_seed))),
            "Share < 0": float((d[column] < 0).mean()),
            "N": int(len(d)),
            "Seeds": int(d.seed.nunique()),
        }
    )


mpc_table = pd.DataFrame(
    {"Impact (transfer quarter)": weighted_mpc(mpc_panel, MPC_IMPACT)}
    | {f"Cumulative, {k}": weighted_mpc(mpc_panel, v) for k, v in MPC_CUMULATIVE.items()}
).T.astype({"N": int, "Seeds": int})
mpc_table.to_csv(OUT / "paper_mpc_table.csv")
mpc_table.style.format(
    {"MPC": "{:.3f}", "SE (seed)": "{:.4f}", "Share < 0": "{:.1%}", "N": "{:,.0f}", "Seeds": "{:.0f}"}
)

In [ ]:
latex = mpc_table[["MPC", "SE (seed)", "Share < 0", "N"]].to_latex(
    float_format="%.3f", header=["MPC", "SE (seed)", "Share $<0$", "$N$"], escape=False
)
(OUT / "paper_mpc_table.tex").write_text(latex)
print(latex)

## 3. MPC by household group
**Primary: the paper's balance-sheet groups** (`classify_balance_sheet_groups`, same rule as the calibration notebook and Table `tab:household_groups`): each household is High/Low on NLA/y, IFA/y and HA/y, High = strictly above the within-seed median, measured in the **baseline run in the quarter before the transfer**. NLA = liquid financial assets − all debt (consumer + mortgage).

Main figure collapses the 8 groups to three (low liquidity = hand-to-mouth):
- **Poor hand-to-mouth:** LLL
- **Wealthy hand-to-mouth:** low NLA, high IFA and/or HA (LLH, LHL, LHH)
- **Doubly wealthy:** high NLA (HLL, HLH, HHL, HHH). HLL holds liquid wealth only and is grouped here as non-hand-to-mouth.

**Robustness: KVW (2014) classes** (`classify_households`), see table 3c. Groups are balance-sheet bins, not MPC determinants.

In [ ]:
groups = classify_balance_sheet_groups(mpc_panel)
mpc_panel = mpc_panel.join(groups[["group_code", "group_label", "group_3"]])


def group_table(panel, column, order):
    rows = {
        g: weighted_mpc(panel[panel[column] == g], MPC_IMPACT).to_dict()
        | {"Population share": float((panel[column] == g).mean())}
        for g in order
    }
    return pd.DataFrame(rows).T[["Population share", "MPC", "SE (seed)", "Share < 0", "N"]].astype({"N": int})


fmt = {"Population share": "{:.1%}", "MPC": "{:.3f}", "SE (seed)": "{:.4f}", "Share < 0": "{:.1%}", "N": "{:,.0f}"}

# 3a. Main figure: three groups.
three_table = group_table(mpc_panel, "group_3", THREE_GROUPS)
three_table.to_csv(OUT / "paper_mpc_by_group3.csv")
fig, ax = plt.subplots(figsize=(5.5, 3.5))
data = [mpc_panel.loc[mpc_panel.group_3 == g, MPC_IMPACT].dropna() for g in THREE_GROUPS]
ax.boxplot(
    data,
    showfliers=False,
    whis=(5, 95),
    patch_artist=True,
    boxprops=dict(facecolor="#7a9cbf"),
    medianprops=dict(color="#1f3b5c", linewidth=1.5),
)
ax.scatter(range(1, 4), [d.mean() for d in data], color="black", zorder=3, s=18, label="Mean")
ax.set_xticks(
    range(1, 4),
    [f"{g.replace(' hand', chr(10) + 'hand')}\n{s:.0%}" for g, s in zip(THREE_GROUPS, three_table["Population share"])],
)
ax.axhline(0, color="grey", linewidth=0.7)
ax.set_ylabel("Impact cash MPC")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "paper_mpc_by_group3.pdf")
fig.savefig(OUT / "paper_mpc_by_group3.png", dpi=200)
plt.show()
three_table.style.format(fmt)

In [ ]:
# 3b. All eight paper groups (table).
eight_table = group_table(mpc_panel, "group_code", list(GROUP_LABELS))
eight_table.insert(0, "Group", [GROUP_LABELS[c] for c in eight_table.index])
eight_table.to_csv(OUT / "paper_mpc_by_group8.csv")
(OUT / "paper_mpc_by_group8.tex").write_text(eight_table.to_latex(float_format="%.3f", escape=False))
eight_table.style.format(fmt)

In [ ]:
# 3c. Robustness: KVW classes (monthly pay, one-month credit-limit proxy, DW = top two quintiles).
mpc_panel["htm_class"] = classify_households(mpc_panel)
kvw_table = group_table(mpc_panel, "htm_class", HTM_CLASSES)
kvw_table.to_csv(OUT / "paper_mpc_by_kvw_class.csv")
kvw_table.style.format(fmt)

## 4. Impulse responses
Mean across seeds (line) and 10th–90th percentile across seeds (band). Deviations from the paired no-shock run: % for real GDP, real consumption and CPI; percentage points for unemployment and the policy rate. Horizon 0 = first quarter with the shock.

In [ ]:
PAPER_IRF_SHOCKS = {
    "policy_rate_100bp": "Policy rate +100bp (4 quarters)",
    "household_transfer_10pct": "One-off transfer (10% median income)",
    # Appendix candidate: "unemp_2pp": "Unemployment +2pp (4 quarters)",
    # Excluded until fixed: income_tax_1pp (payroll wedge), gov_cons_5pct (rationing),
    # *_income_uncertainty_25pct (uncertainty_propensity scale under review).
}
PAPER_IRF_VARIABLES = {
    "real_gdp": "Real GDP, %",
    "real_cons": "Real consumption, %",
    "cpi": "CPI, %",
    "unemp": "Unemployment, pp",
    "policy_rate": "Policy rate, pp",
}
MAX_HORIZON = 16

irf_raw = pd.read_csv(IRF_DIR / "irf_panel.csv")
irf_raw = irf_raw[irf_raw.shock_name.isin(PAPER_IRF_SHOCKS)]
irf = _prepare_irf_plot_panel(irf_raw)
irf = irf[irf.variable.isin(PAPER_IRF_VARIABLES) & (irf.horizon <= MAX_HORIZON)]
bands = (
    irf.groupby(["shock_name", "variable", "horizon"])["plot_value"]
    .agg(mean="mean", p10=lambda x: x.quantile(0.1), p90=lambda x: x.quantile(0.9), n="count")
    .reset_index()
)
bands.to_csv(OUT / "paper_irf_bands.csv", index=False)


def plot_paper_irfs(bands, shocks, variables, path):
    fig, axes = plt.subplots(
        len(shocks), len(variables), figsize=(3.0 * len(variables), 2.4 * len(shocks)), squeeze=False
    )
    for i, (shock, shock_label) in enumerate(shocks.items()):
        for j, (var, var_label) in enumerate(variables.items()):
            ax = axes[i, j]
            d = bands[(bands.shock_name == shock) & (bands.variable == var)]
            ax.fill_between(d.horizon, d.p10, d.p90, color="#7a9cbf", alpha=0.35, linewidth=0)
            ax.plot(d.horizon, d["mean"], color="#1f3b5c", linewidth=1.8)
            ax.axhline(0, color="grey", linewidth=0.7)
            ax.spines[["top", "right"]].set_visible(False)
            if i == 0:
                ax.set_title(var_label, fontsize=10)
            if j == 0:
                ax.set_ylabel(shock_label, fontsize=9)
            if i == len(shocks) - 1:
                ax.set_xlabel("Quarters after shock")
    fig.tight_layout()
    fig.savefig(path.with_suffix(".pdf"))
    fig.savefig(path.with_suffix(".png"), dpi=200)
    plt.show()


plot_paper_irfs(bands, PAPER_IRF_SHOCKS, PAPER_IRF_VARIABLES, OUT / "paper_irf_macro")
print("seeds per cell:", bands.n.unique())

In [ ]:
# Peak responses for the text.
peaks = bands.loc[bands.groupby(["shock_name", "variable"])["mean"].apply(lambda s: s.abs().idxmax())]
peaks[["shock_name", "variable", "horizon", "mean", "p10", "p90"]].round(3)

## 5. Cumulative fiscal multiplier of the one-off transfer
$M_H = \sum_{h=0}^{H-1} \Delta Y^{real}_h \,/\, T$, per seed, then mean and SE across seeds.
- $T$: the transfer, measured as the h0 change in household income (labour incomes are identical across arms at h0).
- Real changes are in h0 prices: each arm is deflated by its own CPI relative to the baseline CPI at h0. No discounting.
- Baseline policy rate is at its floor: no monetary offset (favourable case for fiscal policy).

In [ ]:
TRANSFER_SHOCK = "household_transfer_10pct"
raw = pd.read_csv(IRF_DIR / "irf_panel.csv")
raw = raw[raw.shock_name == TRANSFER_SHOCK]
w = raw.pivot_table(index=["seed", "horizon"], columns="variable", values=["baseline", "shock", "delta"])
cpi_b, cpi_s = w["baseline", "cpi"], w["shock", "cpi"]
cpi_b0 = cpi_b.groupby(level=0).transform("first")
transfer = w["delta", "household_income"].xs(0, level=1)


def real_delta(var):
    return (w["shock", var] / (cpi_s / cpi_b0) - w["baseline", var] / (cpi_b / cpi_b0)).unstack()


rows = {}
for var, label in [("gdp_output", "GDP"), ("household_consumption", "Household consumption (excl. VAT)")]:
    d = real_delta(var)
    for quarters in (1, 4, 8, 12, 16):
        m = d.loc[:, : quarters - 1].sum(axis=1) / transfer
        rows[(label, quarters)] = {"Multiplier": m.mean(), "SE (seed)": m.std(ddof=1) / np.sqrt(len(m))}
multiplier = pd.DataFrame(rows).T
multiplier.index.names = ["Variable", "Quarters"]
multiplier.to_csv(OUT / "paper_transfer_multiplier.csv")
(OUT / "paper_transfer_multiplier.tex").write_text(multiplier.to_latex(float_format="%.2f"))
print(
    f"Transfer = {(transfer / w['baseline', 'household_income'].xs(0, level=1)).mean():.1%} of quarterly household income; seeds = {len(transfer)}"
)
multiplier.round(3)